In [1]:
# =========================
# SETUP + DATA PREP
# =========================
!pip install -q ultralytics

import os
import json
import random
import shutil
from pathlib import Path
from tqdm import tqdm

# =========================
# CONFIG
# =========================
DATA_ROOT = "/kaggle/input/datasets/sonlest/bom-dataset-v3/BOM-Dataset-v3"
ANNOT_PATH = f"{DATA_ROOT}/annotations/annotations_v1.json"
IMAGE_DIR = f"{DATA_ROOT}/images"

OUTPUT_DIR = "/kaggle/working/bom_yolo"
SPLIT_RATIO = (0.6, 0.3, 0.1)

CLASSES = ["PartDrawing", "Note", "Table"]

# =========================
# LOAD COCO
# =========================
with open(ANNOT_PATH) as f:
    coco = json.load(f)

cat_id_to_idx = {cat["id"]: i for i, cat in enumerate(coco["categories"])}

img_to_anns = {}
for ann in coco["annotations"]:
    img_to_anns.setdefault(ann["image_id"], []).append(ann)

images = coco["images"]

# =========================
# SPLIT DATASET
# =========================
random.seed(42)
random.shuffle(images)

n = len(images)
n_train = int(n * SPLIT_RATIO[0])
n_val = int(n * SPLIT_RATIO[1])

splits = {
    "train": images[:n_train],
    "val": images[n_train:n_train+n_val],
    "test": images[n_train+n_val:]
}

# =========================
# CREATE FOLDER
# =========================
for split in splits:
    os.makedirs(f"{OUTPUT_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/labels/{split}", exist_ok=True)

# =========================
# CONVERT TO YOLO
# =========================
def convert_split(img_list, split):
    for img in tqdm(img_list, desc=f"{split}"):
        file_name = img["file_name"]
        src = os.path.join(IMAGE_DIR, file_name)

        if not os.path.exists(src):
            continue

        dst = f"{OUTPUT_DIR}/images/{split}/{file_name}"
        shutil.copy(src, dst)

        w, h = img["width"], img["height"]
        label_path = f"{OUTPUT_DIR}/labels/{split}/{Path(file_name).stem}.txt"

        with open(label_path, "w") as f:
            for ann in img_to_anns.get(img["id"], []):
                x, y, bw, bh = ann["bbox"]
                cls = cat_id_to_idx[ann["category_id"]]

                xc = (x + bw/2) / w
                yc = (y + bh/2) / h
                bw /= w
                bh /= h

                f.write(f"{cls} {xc} {yc} {bw} {bh}\n")

for split in splits:
    convert_split(splits[split], split)

# =========================
# CREATE YAML
# =========================
yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val
test: images/test

names:
  0: PartDrawing
  1: Note
  2: Table
"""

with open("/kaggle/working/bom.yaml", "w") as f:
    f.write(yaml_content)

print("✅ Dataset ready (6/3/1 split)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.4 MB/s eta 0:00:00


test: 100%|██████████| 18/18 [00:00<00:00, 81.90it/s]

✅ Dataset ready (6/3/1 split)


In [2]:
# =========================
# TRAIN + EVALUATE
# =========================
from ultralytics import RTDETR
import torch

# giảm warning không cần thiết
torch.use_deterministic_algorithms(False)

def train_model():
    model = RTDETR("rtdetr-l.pt")

    model.train(
        data="/kaggle/working/bom.yaml",

        # training core
        epochs=100,
        imgsz=1280,
        batch=8,
        device=0,
        workers=4,

        # optimization
        lr0=1e-4,
        patience=15,

        # augmentation (safe for engineering drawings)
        augment=True,
        degrees=2.0,
        translate=0.1,
        scale=0.5,
        mosaic=0.5,

        # logging
        verbose=False,

        project="/kaggle/working/rtdetr",
        name="exp"
    )

    return model.trainer.best


def evaluate_model(weight_path):
    from ultralytics import RTDETR

    model = RTDETR(weight_path)

    metrics = model.val(
        data="/kaggle/working/bom.yaml",
        split="val",
        verbose=False
    )

    print("\n📊 FINAL RESULTS:")
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall: {metrics.box.mr:.4f}")

    return metrics

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# =========================
# MAIN PIPELINE
# =========================

# 1. Train
best_weight = train_model()

# 2. Evaluate
metrics = evaluate_model(best_weight)
print("BEST WEIGHT:", best_weight)

# 3. Inference trên test set
#TEST_DIR = "/kaggle/working/bom_yolo/images/test"
#run_inference(best_weight, TEST_DIR)

print("🚀 ALL DONE!")

Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/bom.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=exp, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective=0.0, pl

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      10.6G     0.8993      13.51     0.9017         15       1280: 100% ━━━━━━━━━━━━ 26/26 1.3s/it 33.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.6s/it 6.6s
                   all         51        227      0.021      0.417     0.0165    0.00949

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100        11G     0.7785      5.115     0.7925         15       1280: 100% ━━━━━━━━━━━━ 26/26 1.0it/s 25.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.1it/s 3.5s
                   all         51        227    0.00509      0.133    0.00753    0.00371

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

     61/100      10.7G     0.2473      0.524     0.2272         17       1280: 100% ━━━━━━━━━━━━ 26/26 1.0it/s 25.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.1it/s 3.5s
                   all         51        227      0.784       0.79      0.832      0.676

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     62/100        11G     0.2836     0.5328     0.2605         12       1280: 100% ━━━━━━━━━━━━ 26/26 1.0it/s 25.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.1it/s 3.5s
                   all         51        227      0.615       0.76      0.755      0.611

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     63/100        11G      0.236     0.4957     0.2285         12       1280: 100% ━━━━━━━━━━━━ 26/26 1.0it/s 25.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.1it/s 3.5s
                   all         51        227      0.749      0.758      0.815      0.669
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 48, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

63 epochs completed in 0.539 hours.
Optimizer stripped from /kaggle/working/rtdetr/exp/weights/last.pt, 66.5MB
Optimizer stripped from /kaggle/working/rtdetr/exp/weights/best.pt, 66.5MB

Validating /kaggle/working/rtdetr/exp/weights/best.pt...
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,989,905 parameters, 0 gradien